## Qwen3-VL-8B Pairwise 24-class Test 추론 및 submission 생성

최종 Pairwise LoRA와 24-class classifier로 Test 819개를 추론하고 `submission.csv`를 저장합니다. Test 데이터의 분포나 특징은 분석하지 않습니다.

### 1. 경로와 추론 설정

`MODEL_RUN_NAME`만 바꾸면 저장된 24-class 계열 모델을 같은 방식으로 비교할 수 있습니다. 현재는 전체 Train으로 학습한 `full_pair_v1/final_adapter`를 사용하며, A100에서는 batch 4를 사용하고 메모리가 부족하면 batch 1로 변경합니다.

In [ ]:
from pathlib import Path
import torch
from google.colab import drive

assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다.'
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/SNU')
DATA_DIR = PROJECT_DIR / 'data'
TEST_CSV = DATA_DIR / 'test.csv'
TEST_IMAGE_DIR = DATA_DIR / 'test'
SAMPLE_SUBMISSION = DATA_DIR / 'sample_submission.csv'

MODEL_NAME = 'Qwen/Qwen3-VL-8B-Instruct'
MODEL_RUN_NAME = 'full_pair_v1'
EXPERIMENT_NAME = MODEL_RUN_NAME
ADAPTER_DIR = (
    PROJECT_DIR / 'outputs' / 'qwen3_vl_8b_lora'
    / MODEL_RUN_NAME / 'final_adapter'
)
CLASSIFIER_PATH = ADAPTER_DIR / 'permutation_classifier.safetensors'
CLASS_MAPPING_PATH = ADAPTER_DIR / 'class_mapping.json'
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'submissions' / EXPERIMENT_NAME
SUBMISSION_PATH = OUTPUT_DIR / 'submission.csv'

INFERENCE_BATCH_SIZE = 4  # RTX 3090에서는 1

for path in (
    TEST_CSV, TEST_IMAGE_DIR, SAMPLE_SUBMISSION, ADAPTER_DIR,
    CLASSIFIER_PATH, CLASS_MAPPING_PATH,
):
    assert path.exists(), f'경로가 없습니다: {path}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Adapter: {ADAPTER_DIR}')
print(f'Classifier: {CLASSIFIER_PATH}')
print(f'Submission: {SUBMISSION_PATH}')

### 2. 필요한 패키지 설치

학습 노트북과 동일한 Transformers·PEFT·Qwen 이미지 처리 버전을 설치합니다.

In [ ]:
%pip install -q transformers==5.13.0 accelerate==1.14.0 peft==0.19.1 qwen-vl-utils==0.0.14 safetensors
%pip uninstall -y torchao

### 3. Test 로드와 학습 프롬프트 정의

Test와 sample submission의 819개 ID 순서가 같은지만 확인합니다. 프롬프트와 chronological order 변환은 학습 및 Validation에서 사용한 것과 동일합니다.

In [ ]:
import json
import re
from itertools import permutations

import pandas as pd
from tqdm.auto import tqdm

test = pd.read_csv(TEST_CSV, dtype=str, encoding='utf-8-sig')
sample_submission = pd.read_csv(
    SAMPLE_SUBMISSION, dtype=str, encoding='utf-8-sig'
)
assert test.columns.tolist() == [
    'Id', 'Input_1', 'Input_2', 'Input_3', 'Input_4', 'Sentence'
]
assert sample_submission.columns.tolist() == ['Id', 'Answer']
assert len(test) == len(sample_submission) == 819
assert test['Id'].tolist() == sample_submission['Id'].tolist()

IDENTITY_ORDER = (1, 2, 3, 4)
with CLASS_MAPPING_PATH.open(encoding='utf-8') as file:
    stored_class_mapping = json.load(file)
CHRONOLOGICAL_CLASSES = tuple(
    tuple(stored_class_mapping['class_index_to_chronological_order'][str(index)])
    for index in range(24)
)
assert len(CHRONOLOGICAL_CLASSES) == 24
assert set(CHRONOLOGICAL_CLASSES) == set(permutations(IDENTITY_ORDER))
CLAUSE_PATTERN = re.compile(
    r'\b(then|after which|after that|and finally|finally|next|before that|followed by|first|before|after)\b,?',
    flags=re.IGNORECASE,
)
STRUCTURED_HINT = (
    'This caption describes distinct sequential stages, often marked by words like '
    "'then', 'after which', or 'finally'. Match each stage of the story to its corresponding image."
)
CONTINUOUS_HINT = (
    'This caption describes a single continuous action or a brief moment. '
    'Focus on subtle visual cues such as camera movement direction, object position changes, '
    'hand/body motion, and lighting changes to infer the temporal order.'
)

def split_clauses(sentence, max_clauses=4):
    sentence = sentence.strip()
    parts = CLAUSE_PATTERN.split(sentence)
    clauses = []
    current = parts[0].strip(' ,;.')
    for index in range(1, len(parts), 2):
        if current:
            clauses.append(current)
        current = parts[index + 1].strip(' ,;.')
    if current:
        clauses.append(current)
    clauses = [clause for clause in clauses if clause]
    if len(clauses) > max_clauses:
        clauses = clauses[:max_clauses - 1] + [' '.join(clauses[max_clauses - 1:])]
    return clauses or [sentence]

def chronological_order_to_answer(order):
    answer = [0, 0, 0, 0]
    for position, image_number in enumerate(order, start=1):
        answer[image_number - 1] = position
    return answer

def class_index_to_answer(class_index):
    return chronological_order_to_answer(
        CHRONOLOGICAL_CLASSES[int(class_index)]
    )

def build_prompt_messages(row):
    sentence = row['Sentence']
    hint = STRUCTURED_HINT if len(split_clauses(sentence)) >= 3 else CONTINUOUS_HINT
    content = []
    for image_number in IDENTITY_ORDER:
        image_path = TEST_IMAGE_DIR / row['Id'] / row[f'Input_{image_number}']
        content.append({'type': 'image', 'image': str(image_path)})
        content.append({'type': 'text', 'text': f'\nImage {image_number}\n'})
    content.append({
        'type': 'text',
        'text': (
            f'Thinking about the sentence: "{sentence}"\n'
            f'{hint}\n'
            'Look at the 4 images above labeled Image 1 to Image 4. '
            'Determine the correct chronological order of these images to match the sentence. '
            'Provide the answer ONLY as a Python list of integers. '
            'Example: [1, 2, 3, 4]'
        ),
    })
    return [{'role': 'user', 'content': content}]

print(f'Test samples: {len(test):,}')
print('24-class mapping 확인 완료')

### 4. 모델·LoRA·24-class classifier 로드

Qwen3-VL-8B 백본, 최종 Pairwise LoRA, 24-class classifier를 추론 모드로 불러옵니다.

In [ ]:
import torch.nn as nn
from safetensors.torch import load_file
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel
from qwen_vl_utils import process_vision_info

processor = AutoProcessor.from_pretrained(MODEL_NAME)
processor.tokenizer.padding_side = 'right'

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map='auto',
    low_cpu_mem_usage=True,
)
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False,
)
model_device = next(model.parameters()).device
text_config = getattr(
    model.get_base_model().config,
    'text_config',
    model.get_base_model().config,
)
hidden_size = int(text_config.hidden_size)
classifier = nn.Linear(hidden_size, 24, bias=True)
classifier.load_state_dict(load_file(str(CLASSIFIER_PATH), device='cpu'))
classifier.to(model_device, dtype=torch.float32)
multimodal_core = model.get_base_model().model
model.config.use_cache = False
model.eval()
classifier.eval()

def pool_last_prompt_token(last_hidden_state, attention_mask):
    last_indices = attention_mask.sum(dim=1).long() - 1
    batch_indices = torch.arange(
        last_hidden_state.shape[0], device=last_hidden_state.device
    )
    return last_hidden_state[batch_indices, last_indices].float()

print('모델·adapter·classifier 로드 완료')

### 5. Test 추론 및 submission.csv 저장

학습 및 Validation과 동일하게 마지막 prompt hidden state를 24-class classifier에 입력합니다. 선택된 chronological class를 공식 Answer 형식으로 변환하고 sample submission의 ID 순서를 유지해 저장합니다.

In [ ]:
predictions = []

for start_index in tqdm(
    range(0, len(test), INFERENCE_BATCH_SIZE),
    desc='Qwen3-VL Test inference',
):
    batch_rows = test.iloc[start_index:start_index + INFERENCE_BATCH_SIZE]
    batch_messages = [
        build_prompt_messages(row)
        for _, row in batch_rows.iterrows()
    ]
    batch_texts = [
        processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        for messages in batch_messages
    ]
    all_images = []
    for messages in batch_messages:
        image_inputs, video_inputs = process_vision_info(messages)
        assert not video_inputs
        all_images.extend(image_inputs)
    inputs = processor(
        text=batch_texts,
        images=all_images,
        padding=True,
        return_tensors='pt',
    ).to(model_device)

    with torch.inference_mode(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16
    ):
        outputs = multimodal_core(
            **inputs, use_cache=False, return_dict=True
        )
    pooled = pool_last_prompt_token(
        outputs.last_hidden_state, inputs['attention_mask']
    )
    with torch.inference_mode():
        logits = classifier(pooled)
        predicted_classes = logits.argmax(dim=-1)
    predictions.extend(
        class_index_to_answer(class_index.item())
        for class_index in predicted_classes
    )

assert len(predictions) == len(test) == 819
assert all(sorted(answer) == [1, 2, 3, 4] for answer in predictions)

submission = sample_submission[['Id']].copy()
submission['Answer'] = [str(answer) for answer in predictions]
submission.to_csv(SUBMISSION_PATH, index=False, encoding='utf-8-sig')

saved_submission = pd.read_csv(
    SUBMISSION_PATH, dtype=str, encoding='utf-8-sig'
)
assert saved_submission.columns.tolist() == ['Id', 'Answer']
assert len(saved_submission) == 819
assert saved_submission['Id'].tolist() == test['Id'].tolist()

print(f'Submission 저장 완료: {SUBMISSION_PATH}')

### 6. submission.csv 내려받기

5번 셀이 완료된 후 실행합니다.

In [ ]:
from google.colab import files
files.download(str(SUBMISSION_PATH))